In [ ]:
from scripts import utils

from pathlib import Path
import numpy as np
import rasterio
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
import yaml
import os

In [ ]:
configs_dir = Path("configs/")
output_dir = Path("output/")

# Use config file stems as valid titles (e.g., my_scenario.yaml -> "my_scenario")
yaml_files = [*configs_dir.glob("*.yaml"), *configs_dir.glob("*.yml")]
config_titles = set()

for p in yaml_files:
    with p.open("r", encoding="utf-8") as f:
        cfg = yaml.safe_load(f) or {}

    title = cfg.get("name")
    if isinstance(title, str) and title.strip():
        config_titles.add(title.strip())
    else:
        print(f"Warning: missing/invalid 'name' in {p.name}")

# Load only GeoTIFFs whose stem exactly matches a config title.
# This excludes suffix variants like "... Existing Protected Areas".
tif_files = [*output_dir.glob("*.tif"), *output_dir.glob("*.tiff")]
selected = sorted(
    {p.stem: p for p in tif_files if p.stem in config_titles}.values(),
    key=lambda p: p.stem
)

if not selected:
    raise FileNotFoundError("No matching GeoTIFFs found in /output for config titles in /configs.")

missing = sorted(config_titles - {p.stem for p in selected})
if missing:
    print(f"Warning: {len(missing)} config titles have no exact-matching GeoTIFF in /output.")

# Read binary masks and validate alignment
masks = []
ref_shape = None
ref_transform = None
ref_crs = None

nodata_mask = None

for fp in selected:
    with rasterio.open(fp) as src:
        arr = src.read(1, masked=True)  # respects nodata/NaN from raster metadata
        valid = ~np.ma.getmaskarray(arr)

        data = np.ma.filled(arr, 0)
        mask = ((data > 0) & valid).astype(np.uint8)  # binary only on valid pixels

        if ref_shape is None:
            ref_shape = mask.shape
            ref_transform = src.transform
            ref_crs = src.crs
            nodata_mask = ~valid
        else:
            if mask.shape != ref_shape or src.transform != ref_transform or src.crs != ref_crs:
                raise ValueError(f"Raster alignment mismatch: {fp.name}")
            nodata_mask |= ~valid  # mask out pixels nodata in any raster

        masks.append(mask)

stack = np.stack(masks, axis=0)              # (n_masks, rows, cols)
consensus = stack.sum(axis=0)                # values: 0..n_masks
consensus = np.ma.array(consensus, mask=nodata_mask)
n_masks = stack.shape[0]

print(f"Loaded {n_masks} masks:")
for p in selected:
    print(f" - {p.name}")

In [ ]:
# Get 1D x/y coordinates for every pixel center from the previously loaded `src`
ds = rasterio.open(src.name) if src.closed else src

rows, cols = np.indices((ds.height, ds.width))
xs, ys = rasterio.transform.xy(ds.transform, rows, cols, offset="center")

x_1d = np.asarray(xs).ravel()
y_1d = np.asarray(ys).ravel()

# print(x_1d.shape, y_1d.shape)  # both should be (ds.height * ds.width,)

if ds is not src:
    ds.close()

In [ ]:
# Categorical consensus visualization: 0,1,2,...,n_masks agreeing
colors = ["#D2D2D2B9"] + [plt.cm.viridis(i / max(n_masks, 1)) for i in range(1, n_masks + 1)]
cmap = ListedColormap(colors)
norm = BoundaryNorm(np.arange(-0.5, n_masks + 1.5, 1), cmap.N)

fig = plt.figure(figsize=(8, 6))
fig.set_constrained_layout(False)
try:
    fig.set_tight_layout(False)
except Exception:
    pass

# Fixed title area (inside the figure)
title_ax = fig.add_axes([0.2, 0.91, 0.65, 0.06])
title_ax.axis("off")
title_text = f"Consensus map across {n_masks} Scenarios"
title_ax.text(0.5, 0.5, title_text, ha="center", va="center", wrap=True)

ax = fig.add_axes([0.10, 0.10, 0.75, 0.75])
# ax.set_title(title_text)

im = ax.imshow(
    consensus,
    extent=[x_1d.min(), x_1d.max(), y_1d.min(), y_1d.max()],
    cmap=cmap,
    norm=norm,
    aspect="auto",
)
ax.set_xlabel("X")
ax.set_ylabel("Y")


cax = fig.add_axes([0.88, 0.10, 0.03, 0.75])  # fixed colorbar area
cbar = fig.colorbar(im, cax=cax, ticks=np.arange(0, n_masks + 1))
cbar.set_label("Number of scenarios agreeing (presence=1)")

ax.set_aspect("equal")

os.makedirs("output", exist_ok=True)
plt.savefig(os.path.join("output", "consensus_map"), dpi=300)

plt.show()

# Optional quick summary of agreement levels
levels, counts = np.unique(consensus, return_counts=True)
print("\nPixel counts by agreement level:")
for lvl, cnt in zip(levels, counts):
    print(f"{lvl}: {cnt}")
    